# Matrix Spike (MS) Detection

**Datamine CCLAS QC Anomaly Detection**

This notebook demonstrates the full Matrix Spike detection workflow, following the same architecture as the SRMS detector. MS samples assess whether the chemical matrix of a real ore sample suppresses or inflates the instrument's ability to detect a specific analyte.

---


## Detection Scope

**Included:** SPK / Spike records (`ANALYTICAL_TYPE = Spike`, `QC_TYPE = MS`).

**Excluded:** BLK, STD, REP, DUP, MSD, and all other record types.

Since `PARENT_NUMERIC_FINAL_VALUE` is entirely null in the current export, recovery percentage cannot be computed. All detection is based on deviation between `NUMERIC_FINAL_VALUE` and `INTERNAL_TARGET_VALUE`.

---


## Detection Architecture

```text
CCLAS Export (QC_Anomaly_Training_Data_v2.xlsx)
      |
      v
Extract MS Records (ANALYTICAL_TYPE = Spike, QC_TYPE = MS)
      |
      v
Feature Engineering
(TARGET_DEVIATION, NORM_DEV, POSITION_IN_RANGE, rolling features, GROUP_MEDIAN, ROBUST_Z_SCORE)
      |
      v
Rule-Based Flag Layer
(Pass / Warning / Failure against CCLAS limits)
      |
      v
Isolation Forest
(multivariate anomaly detection on feature set)
      |
      v
Final Risk Score
(Critical / High / Medium / Low)
      |
      v
Plain-Language Reason + Output CSV
```

Known QC acceptance limits take priority over machine learning. A hard limit failure remains Critical regardless of Isolation Forest signals.

---


## 1. Setup

In [4]:
%pip install scikit-learn

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# ---------------------------------------------------------------------------
# Configuration — equivalent to YAML config in SRMS detector
# ---------------------------------------------------------------------------

CONFIG = {
    "group_cols":              ["STD_LOT_CODE", "SCHEME_CODE", "ANALYTE_CODE"],
    "rolling_window":          10,
    "rolling_min_periods":     3,
    "drift_threshold":         0.5,
    "ignored_statuses":        {"IgnoredUpperFailure", "IgnoredLowerFailure"},
    "mad_scale":               0.6745,
    "robust_z_threshold":      3.0,
    "iforest_contamination":   0.05,
    "iforest_n_estimators":    200,
    "iforest_random_state":    42,
}

config_table = pd.DataFrame(CONFIG.items(), columns=["Setting", "Value"])
display(config_table)


  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 10.9 MB/s  0:00:00 eta 0:00:01
Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl (20.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


,Setting,Value
0,group_cols,"[STD_LOT_CODE, SCHEME_CODE, ANALYTE_CODE]"
1,rolling_window,10
2,rolling_min_periods,3
3,drift_threshold,0.5
4,ignored_statuses,"{IgnoredUpperFailure, IgnoredLowerFailure}"
5,mad_scale,0.6745
6,robust_z_threshold,3.0
7,iforest_contamination,0.05
8,iforest_n_estimators,200
9,iforest_random_state,42


## 2. Load MS Data

In [8]:
DATA_PATH = Path("data/raw/QC_Anomaly_Training_Data_v2.xlsx")

if not DATA_PATH.exists():
    DATA_PATH = Path.home() / "Downloads" / "QC_Anomaly_Training_Data_v2.xlsx"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Training data not found: {DATA_PATH}")

df_raw = pd.read_excel(DATA_PATH, sheet_name="SPK(MS) Assessment", engine="openpyxl")
df_raw.columns = [str(c).strip().upper() for c in df_raw.columns]

# Drop unnamed Excel columns
unnamed = [c for c in df_raw.columns if c.strip() == "" or c.startswith("UNNAMED")]
if unnamed:
    df_raw = df_raw.drop(columns=unnamed)

print(f"Loaded {len(df_raw)} MS rows, {len(df_raw.columns)} columns.")
print(f"ANALYTICAL_TYPE: {df_raw['ANALYTICAL_TYPE'].value_counts().to_dict()}")
print(f"QC_TYPE:         {df_raw['QC_TYPE'].value_counts().to_dict()}")
print(f"STANDARD_STATUS: {df_raw['STANDARD_STATUS'].value_counts().to_dict()}")
print(f"STD_LOT_CODE:    {df_raw['STD_LOT_CODE'].value_counts().to_dict()}")
print(f"Unique analytes: {df_raw['ANALYTE_CODE'].nunique()}")
print(f"PARENT_NUMERIC_FINAL_VALUE non-null: {df_raw['PARENT_NUMERIC_FINAL_VALUE'].notna().sum()} (recovery not computable)")


Loaded 6958 MS rows, 28 columns.
ANALYTICAL_TYPE: {'Spike': 6958}
QC_TYPE:         {'MS': 6958}
STANDARD_STATUS: {'Pass': 4748, 'UpperWarning': 758, 'LowerWarning': 594, 'LowerFailure': 250, 'UpperFailure': 230, 'IgnoredUpperFailure': 224, 'IgnoredLowerFailure': 154}
STD_LOT_CODE:    {'OREAS_502C': 6958}
Unique analytes: 57
PARENT_NUMERIC_FINAL_VALUE non-null: 0 (recovery not computable)


## 3. Prepare MS Records

Coerce numeric types, parse dates, mark ignored records, and sort chronologically within each group. Ignored records are retained in the output but excluded from group baselines.


In [9]:
NUMERIC_COLS = [
    "NUMERIC_FINAL_VALUE", "INTERNAL_TARGET_VALUE",
    "INTERNAL_MIN_VALUE", "INTERNAL_MAX_VALUE",
    "INTERNAL_MIN_WARNING_VALUE", "INTERNAL_MAX_WARNING_VALUE",
]

df = df_raw.copy()
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["ANALYSED_DATE"] = pd.to_datetime(df["ANALYSED_DATE"], errors="coerce")

df["IS_IGNORED"] = df["STANDARD_STATUS"].isin(CONFIG["ignored_statuses"])

df = df.sort_values(CONFIG["group_cols"] + ["ANALYSED_DATE"]).reset_index(drop=True)

print(f"MS records prepared: {len(df)} rows")
print(f"Ignored records:     {df['IS_IGNORED'].sum()} ({df['IS_IGNORED'].mean()*100:.1f}%)")
print(f"Negative values:     {(df['NUMERIC_FINAL_VALUE'] < 0).sum()} (retained — valid near detection limit)")


MS records prepared: 6958 rows
Ignored records:     378 (5.4%)
Negative values:     298 (retained — valid near detection limit)


## 4. Feature Engineering

Derives 14 features across four groups: deviation from target, normalised deviation, limit position features, and temporal rolling features. All features are computed within `STD_LOT_CODE × SCHEME_CODE × ANALYTE_CODE` groups using only non-ignored records for baselines.


In [10]:
GROUP_COLS = CONFIG["group_cols"]
ROLLING_WINDOW = CONFIG["rolling_window"]
ROLLING_MIN = CONFIG["rolling_min_periods"]
MAD_SCALE = CONFIG["mad_scale"]

# 1-2. Deviation features
df["TARGET_DEVIATION"] = df["NUMERIC_FINAL_VALUE"] - df["INTERNAL_TARGET_VALUE"]
df["TARGET_DEVIATION_PCT"] = np.where(
    df["INTERNAL_TARGET_VALUE"].abs() > 1e-9,
    (df["TARGET_DEVIATION"] / df["INTERNAL_TARGET_VALUE"]) * 100,
    np.nan
)

# 3-4. Normalised deviation score
upper_span = df["INTERNAL_MAX_WARNING_VALUE"] - df["INTERNAL_TARGET_VALUE"]
lower_span = df["INTERNAL_TARGET_VALUE"] - df["INTERNAL_MIN_WARNING_VALUE"]
df["NORM_DEV"] = np.where(
    df["NUMERIC_FINAL_VALUE"] >= df["INTERNAL_TARGET_VALUE"],
    np.where(upper_span.abs() > 1e-9, df["TARGET_DEVIATION"] / upper_span, np.nan),
    np.where(lower_span.abs() > 1e-9, df["TARGET_DEVIATION"] / lower_span, np.nan)
)
df["ABS_NORM_DEV"] = df["NORM_DEV"].abs()

# 5-8. Limit position features
span = df["INTERNAL_MAX_VALUE"] - df["INTERNAL_MIN_VALUE"]
df["DISTANCE_TO_UPPER_FAILURE"] = df["INTERNAL_MAX_VALUE"] - df["NUMERIC_FINAL_VALUE"]
df["DISTANCE_TO_LOWER_FAILURE"] = df["NUMERIC_FINAL_VALUE"] - df["INTERNAL_MIN_VALUE"]
df["LIMIT_SPAN"] = span
df["POSITION_IN_RANGE"] = np.where(
    span.abs() > 1e-9,
    (df["NUMERIC_FINAL_VALUE"] - df["INTERNAL_MIN_VALUE"]) / span,
    np.nan
)

# 9-11. Rolling temporal features
df["ROLLING_MEAN_NORM_DEV"] = df.groupby(GROUP_COLS)["NORM_DEV"].transform(
    lambda x: x.rolling(ROLLING_WINDOW, ROLLING_MIN).mean())
df["ROLLING_STD_NORM_DEV"] = df.groupby(GROUP_COLS)["NORM_DEV"].transform(
    lambda x: x.rolling(ROLLING_WINDOW, ROLLING_MIN).std())
df["ROLLING_BREACH_COUNT"] = df.groupby(GROUP_COLS)["NORM_DEV"].transform(
    lambda x: (x.abs() > 1).astype(float).rolling(ROLLING_WINDOW, ROLLING_MIN).sum())

# 12-13. Group baseline (non-ignored only)
baseline = (
    df[~df["IS_IGNORED"]].groupby(GROUP_COLS)["NORM_DEV"]
    .median().rename("GROUP_MEDIAN")
)
df = df.join(baseline, on=GROUP_COLS)
df["ABS_DEV_FROM_MEDIAN"] = (df["NORM_DEV"] - df["GROUP_MEDIAN"]).abs()
group_mad = (
    df[~df["IS_IGNORED"]].groupby(GROUP_COLS)["ABS_DEV_FROM_MEDIAN"]
    .median().rename("GROUP_MAD")
)
df = df.join(group_mad, on=GROUP_COLS)

# 14. Robust Z-score
df["ROBUST_Z_SCORE"] = np.where(
    df["GROUP_MAD"] > 1e-9,
    MAD_SCALE * (df["NORM_DEV"] - df["GROUP_MEDIAN"]) / df["GROUP_MAD"],
    np.nan
)

FEATURE_COLS = [
    "TARGET_DEVIATION", "TARGET_DEVIATION_PCT",
    "NORM_DEV", "ABS_NORM_DEV",
    "DISTANCE_TO_UPPER_FAILURE", "DISTANCE_TO_LOWER_FAILURE",
    "LIMIT_SPAN", "POSITION_IN_RANGE",
    "ROLLING_MEAN_NORM_DEV", "ROLLING_STD_NORM_DEV", "ROLLING_BREACH_COUNT",
    "GROUP_MEDIAN", "GROUP_MAD", "ROBUST_Z_SCORE",
]

print("Feature engineering complete.")
print(f"Features derived: {len(FEATURE_COLS)}")
print()
print("Null counts per feature:")
for col in FEATURE_COLS:
    print(f"  {col:<30} {df[col].isna().sum():>5} nulls")


Feature engineering complete.
Features derived: 14

Null counts per feature:
  TARGET_DEVIATION                   0 nulls
  TARGET_DEVIATION_PCT               0 nulls
  NORM_DEV                           0 nulls
  ABS_NORM_DEV                       0 nulls
  DISTANCE_TO_UPPER_FAILURE          0 nulls
  DISTANCE_TO_LOWER_FAILURE          0 nulls
  LIMIT_SPAN                         0 nulls
  POSITION_IN_RANGE                  0 nulls
  ROLLING_MEAN_NORM_DEV            258 nulls
  ROLLING_STD_NORM_DEV             258 nulls
  ROLLING_BREACH_COUNT             258 nulls
  GROUP_MEDIAN                      86 nulls
  GROUP_MAD                         86 nulls
  ROBUST_Z_SCORE                    86 nulls


## 5. Rule-Based Flag Layer

Classifies each record as Failure, Warning, Pass, or Ignored based on CCLAS failure and warning bounds, respecting limit inclusivity flags. This layer is the primary detection signal. A hard limit failure remains Critical in the final risk score regardless of ML signals.


In [ ]:
# ---------------------------------------------------------------------------
# Transform MS values to relative scale (following LCS historic drift approach)
# Target = 0, Failure limits = ±1, Warning limits proportionally between 0 and ±1
# ---------------------------------------------------------------------------

def transform_ms_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply piecewise transform per row using that row's own target and limits.
    Mirrors transform_lcs_values() from LCS_drift_detection_historic.ipynb.

    TRANSFORMED_VALUE interpretation:
        0    = exactly on target
       +1    = at upper failure limit
       -1    = at lower failure limit
    Values outside ±1 are outside failure limits.
    Warning limits transform proportionally between 0 and ±1.
    """
    out = df.copy()
    target     = out["INTERNAL_TARGET_VALUE"]
    upper_span = out["INTERNAL_MAX_VALUE"] - target
    lower_span = target - out["INTERNAL_MIN_VALUE"]

    value = out["NUMERIC_FINAL_VALUE"]
    span_for_value = pd.Series(
    np.where(value >= target, upper_span, lower_span),
    index=df.index
)

    out["TRANSFORMED_VALUE"] = np.where(
        span_for_value.abs() > 1e-9,
        (value - target) / span_for_value,
        np.nan
    )

    out["TRANSFORMED_TARGET"]      = 0.0
    out["TRANSFORMED_MAX"]         = 1.0
    out["TRANSFORMED_MIN"]         = -1.0
    out["TRANSFORMED_MAX_WARNING"] = np.where(
        upper_span.abs() > 1e-9,
        (out["INTERNAL_MAX_WARNING_VALUE"] - target) / upper_span,
        np.nan
    )
    out["TRANSFORMED_MIN_WARNING"] = np.where(
        lower_span.abs() > 1e-9,
        (out["INTERNAL_MIN_WARNING_VALUE"] - target) / lower_span,
        np.nan
    )

    return out


df = transform_ms_values(df)


# ---------------------------------------------------------------------------
# Rule-based flag layer using TRANSFORMED_VALUE (relative scale)
# ---------------------------------------------------------------------------

def _is_inclusive(series, col):
    val = series.get(col, "Y")
    if pd.isna(val):
        return True
    return str(val).strip().upper() == "Y"


def apply_rule_based_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Classify each MS record as Failure, Warning, Pass, or Ignored
    using TRANSFORMED_VALUE against the relative ±1 failure boundaries
    and proportionally transformed warning boundaries.

    Uses limit inclusivity flags consistent with Jurriaan's LCS approach.

    Outputs:
        RULE_FLAG        — Pass / Warning / Failure / Ignored
        RULE_FLAG_COLOUR — green / yellow / red / grey
        RULE_FLAG_REASON — plain-language explanation
    """
    df = df.copy()

    val       = df["TRANSFORMED_VALUE"]
    max_warn  = df["TRANSFORMED_MAX_WARNING"]
    min_warn  = df["TRANSFORMED_MIN_WARNING"]
    is_ignored = df["IS_IGNORED"]

    # Inclusivity flags — default to inclusive if missing
    max_incl      = df.get("INTERNAL_MAX_INCLUSIVE", pd.Series("Y", index=df.index)).fillna("Y")
    min_incl      = df.get("INTERNAL_MIN_INCLUSIVE", pd.Series("Y", index=df.index)).fillna("Y")
    max_warn_incl = df.get("INTERNAL_MAX_WARNING_INCLUSIVE", pd.Series("Y", index=df.index)).fillna("Y")
    min_warn_incl = df.get("INTERNAL_MIN_WARNING_INCLUSIVE", pd.Series("Y", index=df.index)).fillna("Y")

    # Failure: outside ±1 (transformed failure boundaries)
    above_max_fail = np.where(
        (max_incl.str.upper() == "Y"), val >= 1.0, val > 1.0
    )
    below_min_fail = np.where(
        (min_incl.str.upper() == "Y"), val <= -1.0, val < -1.0
    )
    is_failure = above_max_fail | below_min_fail

    # Warning: outside transformed warning boundary but within failure boundary
    above_max_warn = np.where(
        (max_warn_incl.str.upper() == "Y"), val >= max_warn, val > max_warn
    )
    below_min_warn = np.where(
        (min_warn_incl.str.upper() == "Y"), val <= min_warn, val < min_warn
    )
    is_warning = (~is_failure) & (above_max_warn | below_min_warn)

    df["RULE_FLAG"] = np.select(
        [is_ignored, is_failure, is_warning],
        ["Ignored", "Failure", "Warning"],
        default="Pass"
    )

    df["RULE_FLAG_COLOUR"] = np.select(
        [is_ignored, is_failure, is_warning],
        ["grey", "red", "yellow"],
        default="green"
    )

    df["RULE_FLAG_REASON"] = np.select(
        [
            is_ignored,
            below_min_fail & ~is_ignored,
            above_max_fail & ~is_ignored,
            below_min_warn & ~is_failure & ~is_ignored,
            above_max_warn & ~is_failure & ~is_ignored,
        ],
        [
            "Result manually ignored in CCLAS — excluded from modelling baselines",
            "Transformed value (" + val.round(4).astype(str) + ") below lower failure boundary (-1.0) - LowerFailure",
            "Transformed value (" + val.round(4).astype(str) + ") above upper failure boundary (1.0) - UpperFailure",
            "Transformed value (" + val.round(4).astype(str) + ") below lower warning boundary (" + min_warn.round(4).astype(str) + ") - LowerWarning",
            "Transformed value (" + val.round(4).astype(str) + ") above upper warning boundary (" + max_warn.round(4).astype(str) + ") - UpperWarning",
        ],
        default="Transformed value (" + val.round(4).astype(str) + ") within warning limits — Pass"
    )

    return df


df = apply_rule_based_flags(df)

# Validate against CCLAS
non_ignored = df[~df["IS_IGNORED"]].copy()
non_ignored["CCLAS_SIMPLIFIED"] = np.select(
    [
        non_ignored["STANDARD_STATUS"].isin(["UpperFailure", "LowerFailure"]),
        non_ignored["STANDARD_STATUS"].isin(["UpperWarning", "LowerWarning"]),
    ],
    ["Failure", "Warning"],
    default="Pass"
)
agreement = (non_ignored["RULE_FLAG"] == non_ignored["CCLAS_SIMPLIFIED"]).mean()

flag_counts = df["RULE_FLAG"].value_counts()
print("Rule-based flag distribution (relative scale):")
for flag, count in flag_counts.items():
    print(f"  {flag:<10} {count:>5}  ({count/len(df)*100:.1f}%)")
print()
print(f"Agreement with CCLAS STANDARD_STATUS (non-ignored): {agreement*100:.1f}%")
print()
print("Transformed value sample:")
print(df[["ANALYTE_CODE", "NUMERIC_FINAL_VALUE", "INTERNAL_TARGET_VALUE",
          "TRANSFORMED_VALUE", "TRANSFORMED_MAX_WARNING", "TRANSFORMED_MIN_WARNING",
          "STANDARD_STATUS", "RULE_FLAG", "RULE_FLAG_REASON"]].head(10).to_string())

Rule-based flag distribution (relative scale):
  Pass        4748  (68.2%)
  Warning     1352  (19.4%)
  Failure      480  (6.9%)
  Ignored      378  (5.4%)

Agreement with CCLAS STANDARD_STATUS (non-ignored): 100.0%

Transformed value sample:
  ANALYTE_CODE  NUMERIC_FINAL_VALUE  INTERNAL_TARGET_VALUE  TRANSFORMED_VALUE  TRANSFORMED_MAX_WARNING  TRANSFORMED_MIN_WARNING STANDARD_STATUS RULE_FLAG                                                                 RULE_FLAG_REASON
0           AG             1.234450                  0.779           0.342989                 0.634284                -0.634284            Pass      Pass                           Transformed value (0.343) within warning limits — Pass
1           AG             1.543689                  0.779           0.575871                 0.634284                -0.634284            Pass      Pass                          Transformed value (0.5759) within warning limits — Pass
2           AG             1.650000                

## 6. Isolation Forest

Applies Isolation Forest on the feature set to detect multivariate anomalies, results that are statistically unusual in combination even when no individual rule threshold is crossed. This provides an additional detection signal that does not override known limit failures.


In [ ]:
IF_FEATURES = [
    "NORM_DEV",
    "ABS_NORM_DEV",
    "POSITION_IN_RANGE",
    "ROLLING_MEAN_NORM_DEV",
    "ROLLING_STD_NORM_DEV",
    "ROLLING_BREACH_COUNT",
    "ROBUST_Z_SCORE",
    "DISTANCE_TO_UPPER_FAILURE",
    "DISTANCE_TO_LOWER_FAILURE",
]

# Train on non-ignored records only — consistent with SRMS detector
train_mask = ~df["IS_IGNORED"]
X = df[IF_FEATURES].copy()
X_train = X[train_mask].copy()

imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_all_imputed   = imputer.transform(X)

iforest = IsolationForest(
    n_estimators=CONFIG["iforest_n_estimators"],
    contamination=CONFIG["iforest_contamination"],
    random_state=CONFIG["iforest_random_state"],
    n_jobs=-1,
)
iforest.fit(X_train_imputed)

df["IF_SCORE"]   = -iforest.score_samples(X_all_imputed)
df["IF_ANOMALY"] = iforest.predict(X_all_imputed) == -1

df["IF_STATUS"] = np.where(df["IF_ANOMALY"], "WARNING", "PASS")

print(f"Isolation Forest complete.")
print(f"  Features used:   {IF_FEATURES}")
print(f"  Contamination:   {CONFIG['iforest_contamination']}")
print(f"  Trained on:      {train_mask.sum()} non-ignored records")
print(f"  IF anomalies:    {df['IF_ANOMALY'].sum()} ({df['IF_ANOMALY'].mean()*100:.1f}%)")
print()
display(df[["ANALYTE_CODE", "SCHEME_CODE", "NUMERIC_FINAL_VALUE",
            "RULE_FLAG", "IF_SCORE", "IF_ANOMALY", "IF_STATUS"]].head(10))


Isolation Forest complete.
  Features used:   ['NORM_DEV', 'ABS_NORM_DEV', 'POSITION_IN_RANGE', 'ROLLING_MEAN_NORM_DEV', 'ROLLING_STD_NORM_DEV', 'ROLLING_BREACH_COUNT', 'ROBUST_Z_SCORE', 'DISTANCE_TO_UPPER_FAILURE', 'DISTANCE_TO_LOWER_FAILURE']
  Contamination:   0.05
  Trained on:      6580 non-ignored records
  IF anomalies:    505 (7.3%)



,ANALYTE_CODE,SCHEME_CODE,NUMERIC_FINAL_VALUE,RULE_FLAG,IF_SCORE,IF_ANOMALY,IF_STATUS
0,AG,GE_ICP40Q12,1.234450,Pass,0.332634,False,PASS
1,AG,GE_ICP40Q12,1.543689,Pass,0.346808,False,PASS
2,AG,GE_ICP40Q12,1.650000,Warning,0.387566,False,PASS
3,AG,GE_ICP40Q12,1.275510,Pass,0.352061,False,PASS
4,AG,GE_ICP40Q12,1.661376,Warning,0.379741,False,PASS
5,AG,GE_ICP40Q12,1.160000,Pass,0.342765,False,PASS
6,AG,GE_ICP40Q12,0.720000,Pass,0.371547,False,PASS
7,AG,GE_ICP40Q12,1.341176,Pass,0.345183,False,PASS
8,AG,GE_ICP40Q12,1.248731,Pass,0.343092,False,PASS
9,AG,GE_ICP40Q12,1.428571,Pass,0.348731,False,PASS
